# AgentOps Lab 06 - Human-in-the-loop and permissions

This notebook introduces consequence. The agent can investigate freely with read-only tools, but it cannot restart services, roll back deployments, or notify customers without a human approval boundary.

The goal is least privilege, not a pure security lecture: give the agent enough permission to be useful, but pause before irreversible or customer-visible actions.


## Approval flow

```mermaid
flowchart TD
    A["Agent"] --> B["Proposes restart checkout-api"]
    B --> C["Policy check"]
    C --> D{"Requires approval?"}
    D -- "No" --> E["Execute read/propose action"]
    D -- "Yes" --> F["Persist paused state"]
    F --> G["Human review"]
    G --> H{"Decision"}
    H -- "approve" --> I["Resume and execute"]
    H -- "modify" --> J["Resume with edited action"]
    H -- "reject" --> K["Resume with rejection and stop/escalate"]
```

LangGraph/LangChain human-in-the-loop patterns support pausing before selected tool actions and resuming from persisted state after approval, modification, or rejection. This lab uses a dependency-free checkpoint store so the concept is runnable without infrastructure.


## Permission levels

| Level | Examples | Purpose |
| --- | --- | --- |
| READ | query logs, retrieve runbooks, inspect deployments | Gather evidence without side effects |
| PROPOSE | prepare rollback, draft notification, prepare ticket | Create reviewable artifacts without execution |
| EXECUTE WITH APPROVAL | restart, rollback, send notification | Perform consequential actions only after human approval |

A useful policy is simple enough to inspect:

```python
approval_policy = {
    "query_logs": False,
    "get_status": False,
    "restart_service": True,
    "rollback_deployment": True,
    "notify_customers": True,
}
```


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root / "labs"))

from agentops_lab.human_permissions import (
    APPROVAL_POLICY,
    InMemoryCheckpointStore,
    ProposedAction,
    demo_approval_paths,
    permission_matrix,
    propose_restart_after_investigation,
    requires_approval,
    resume_with_human_decision,
)


## Inspect the permission matrix

The policy separates evidence gathering from proposed actions and consequential execution.


In [ ]:
permission_matrix()


## Pause before restart

The agent investigates checkout, proposes `restart_service`, and the policy pauses the run before execution. The paused run stores evidence plus the proposed action so the human reviewer is not asked a context-free "Approve?" question.


In [ ]:
store = InMemoryCheckpointStore()
paused = propose_restart_after_investigation(store)
print(paused.status)
print(paused.proposed_action)
print("requires approval:", requires_approval(paused.proposed_action))
print("evidence items:", len(paused.evidence))


## Resume after approval

Approval resumes from persisted state and executes the exact proposed action. In production, the audit event should include actor identity, timestamp, evidence, original action, final action, and reason.


In [ ]:
approved = resume_with_human_decision(store, paused.run_id, "approve", reason="Incident commander approved restart after checking evidence.")
print(approved.status)
print(approved.message)
print(approved.audit)


## Modify or reject

Human review is more useful when it supports approve, modify, and reject. Modification lets the reviewer narrow scope; rejection stops execution and sends the agent back to investigation or escalation.


In [ ]:
demo = demo_approval_paths()
print("modified:", demo["modified"].status, demo["modified"].audit.final_action)
print("rejected:", demo["rejected"].status, demo["rejected"].message)


## Optional LangGraph/LangChain implementation shape

When porting this to a real graph runtime, place the policy gate immediately before selected tool actions. Persist the graph state when interrupted, then resume with a human command that approves, edits, or rejects the pending tool call.

```python
approval_policy = {
    "query_logs": False,
    "get_status": False,
    "restart_service": True,
    "rollback_deployment": True,
    "notify_customers": True,
}

def before_tool(state):
    action = state["proposed_action"]
    if approval_policy.get(action.tool, True):
        interrupt({"action": action, "evidence": state["evidence"]})
    return state
```

The exact API depends on the runtime version, but the design rule is stable: do not let a prompt be the only thing between a model and a consequential operation.


## Exercises

- Add a `rollback_deployment` proposal and require approval before execution.
- Add `send_notification` and require the reviewer to edit customer-facing wording before approval.
- Add actor identity and timestamps to the audit event.
- Create a permission matrix for support agent, on-call agent, and incident commander agent.
- Decide which actions should be unavailable rather than merely approval-gated.

References: [LangGraph human-in-the-loop](https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/), [LangGraph persistence](https://langchain-ai.github.io/langgraph/concepts/persistence/), [OpenAI Agents SDK guardrails](https://openai.github.io/openai-agents-python/guardrails/), and [Building AI Agents: From Loops to Teams](https://www.linkedin.com/pulse/building-ai-agents-from-loops-teams-oneplusi-y3atc/).
